# FREE DAILY VIDEO ENGINE — Kobe (animal lip-sync) + Brendan (human)

**$0. One Colab T4 session produces both clips.** Routes around HF's ~300s/day ZeroGPU cap
(LatentSync Spaces reserve 180s *per call*; a Colab T4 session has no such per-call budget).

| Phase | Model | Who | Why |
|---|---|---|---|
| A | **JoyVASA** (`animation_mode animal`) | 🐶 Kobe | Only free model trained for **animal** faces |
| B | **LatentSync** | 🧑 Brendan | Highest-quality open lip-sync (human) |

**Runtime → Change runtime type → T4 GPU → Run all.**

*Verified 2026-09-05 against the real repos: JoyVASA's CLI is `--reference/--audio/--animation_mode animal`
(not `--source_image/--driven_audio` — that's an AI-guide hallucination).*

---
## Before you run: upload 4 files to the Colab Files panel
- `kobe.png` — clear front-facing Kobe (use `Brands/Kobe/refs/kobe-viewsai.png`)
- `kobe.wav` — Kobe's 8–10s line (Kokoro TTS → wav)
- `brendan.mp4` — short clip of Brendan (or skip Phase B)
- `brendan.wav` — Brendan's line

## Setup (once per session, ~5 min)

In [ ]:
!apt-get install -y ffmpeg git-lfs > /dev/null 2>&1
!git lfs install --skip-repo > /dev/null 2>&1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print('✓ GPU ready')

In [ ]:
from google.colab import files
print('Upload: kobe.png, kobe.wav  (and brendan.mp4, brendan.wav for Phase B)')
files.upload()

# PHASE A — KOBE (JoyVASA, animal mode) 🐶

JoyVASA = audio → identity-independent facial motion (diffusion) → rendered via LivePortrait.
The `--animation_mode animal` flag is what makes it work on a Pomeranian snout where
human-face models (LatentSync/EchoMimic/MuseTalk) fail.

In [ ]:
%cd /content
!git clone -q https://github.com/jdh-algo/JoyVASA.git
%cd /content/JoyVASA
!pip install -q tyro==0.8.5 accelerate==0.28.0 bitsandbytes==0.43.1 diffusers==0.27.2 \
    einops==0.8.0 librosa==0.10.2.post1 mediapipe==0.10.14 imageio-ffmpeg pykalman \
    opencv-python scipy scikit-image onnxruntime-gpu soundfile
print('✓ JoyVASA deps')

In [ ]:
import os
os.makedirs('/content/JoyVASA/pretrained_weights', exist_ok=True)
%cd /content/JoyVASA
# JoyVASA + audio encoder + LivePortrait renderer (~10GB)
!git clone -q https://huggingface.co/jdh-algo/JoyVASA pretrained_weights/JoyVASA
!git clone -q https://huggingface.co/facebook/wav2vec2-base-960h pretrained_weights/wav2vec2-base-960h
!git clone -q https://huggingface.co/KwaiVGI/LivePortrait pretrained_weights/liveportrait
!ls pretrained_weights
print('✓ checkpoints')

In [ ]:
# ---- KOBE INFERENCE (animal mode) ----
%cd /content/JoyVASA
!python inference.py \
  --reference /content/kobe.png \
  --audio /content/kobe.wav \
  --animation_mode animal \
  --output_dir /content/outputs/kobe \
  --flag_stitching False

import glob, shutil, os
vids = sorted(glob.glob('/content/outputs/kobe/**/*.mp4', recursive=True), key=os.path.getmtime)
if vids:
    shutil.copy(vids[-1], '/content/kobe_talking.mp4')
    print('✓ KOBE CLIP:', vids[-1])
    from google.colab import files as f2; f2.download('/content/kobe_talking.mp4')
else:
    print('✗ no output — check the log above')

### If JoyVASA errors on the animal path
Try in this order (each is a one-line change in the cell above):
1. `--animation_mode human` (sometimes the animal flag needs the human renderer to init first)
2. `--flag_use_half_precision True` (T4 supports fp16; faster)
3. `--flag_pasteback True` if you want the full original frame back instead of the face crop

# PHASE B — BRENDAN (LatentSync, human) 🧑

Highest-quality open lip-sync. Runs on the same T4 — **no HF ZeroGPU quota involved**,
which is exactly why we do it here instead of the free Space (180s/call, ~300s/day cap).

In [ ]:
%cd /content
!git clone -q https://github.com/bytedance/LatentSync.git
%cd /content/LatentSync
!pip install -q -r requirements.txt 2>&1 | tail -2
!bash setup_env.sh 2>&1 | tail -3
print('✓ LatentSync ready')

In [ ]:
# Download LatentSync 1.6 checkpoints
%cd /content/LatentSync
!huggingface-cli download ByteDance/LatentSync-1.6 --local-dir checkpoints 2>&1 | tail -2
import os
os.makedirs('configs/unet', exist_ok=True)
!wget -q -O configs/unet/second_stage.yaml https://raw.githubusercontent.com/bytedance/LatentSync/main/configs/unet/second_stage.yaml || true
print('✓ checkpoints')

In [ ]:
# ---- BRENDAN INFERENCE ----
%cd /content/LatentSync
!python -m scripts.inference \
  --unet_config_path "configs/unet/second_stage.yaml" \
  --inference_ckpt_path "checkpoints/latentsync_unet.pt" \
  --inference_steps 20 \
  --guidance_scale 1.5 \
  --video_path /content/brendan.mp4 \
  --audio_path /content/brendan.wav \
  --video_out_path /content/brendan_out.mp4

import os, shutil
if os.path.exists('/content/brendan_out.mp4'):
    shutil.copy('/content/brendan_out.mp4', '/content/brendan_talking.mp4')
    print('✓ BRENDAN CLIP ready')
    from google.colab import files as f3; f3.download('/content/brendan_talking.mp4')

---
# Finish on your Mac (free, local)
```bash
# drop the downloaded clips into the movement library, then:
python3 ~/ViewsOSComplete/scripts/free-stack-videos/assemble_kobe.py \
  --movement kobe_talking \
  --line "Drop AI in the comments and I'll send you the whole system" \
  --title "AI recruiting. 24/7." --cta "Comment AI"
```
→ Kokoro voice + safe-zone text → `platform_render.py` → 12 platform-ready files.

**Daily loop:** Run all (~8-10 min) → download 2 clips → assemble → publish.